# Pipeline de Detección de Comentarios Tóxicos

Este notebook documenta todo el proceso seguido: desde el dataset hasta la evaluación de los modelos.

**Rama**: `feature/toxic-filter`  
**Dataset**: `youtoxic_english_1000.csv` (1000 comentarios de YouTube en inglés)  
**Objetivo**: clasificar comentarios como tóxicos o no tóxicos

## 1. Dataset

**Archivo**: `data/raw/youtoxic_english_1000.csv`

- 1000 filas, 15 columnas
- Columna principal: `Text` (comentario), `IsToxic` (etiqueta binaria)
- Etiquetas adicionales: `IsAbusive`, `IsThreat`, `IsProvocative`, `IsObscene`, `IsHatespeech`, `IsRacist`, `IsNationalist`, `IsSexist`, `IsHomophobic`, `IsReligiousHate`, `IsRadicalism`
- Las etiquetas vienen como booleanos Python o strings `TRUE`/`FALSE` — se normalizan con `.astype(str).str.upper().map({'TRUE':1,'FALSE':0})`

**Distribución de clases**:
- No tóxico: ~540 (54%)
- Tóxico: ~460 (46%)

Dataset relativamente balanceado, aunque se usó `class_weight='balanced'` en el baseline por precaución.

## 2. Preprocesamiento

**Script**: `src/data/prepare_dataset.py`

### Limpieza de texto (`clean_text`)

```python
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)   # elimina URLs
    text = re.sub(r'@\w+', '', text)              # elimina menciones
    text = re.sub(r'[^a-z\s]', '', text)          # elimina puntuación y números
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words]
    return ' '.join(tokens)
```

Pasos: lowercase → quitar URLs → quitar menciones → quitar puntuación → quitar stopwords → lematización (NLTK WordNetLemmatizer)

### Split

| Conjunto | Filas | % |
|---|---|---|
| Train | 700 | 70% |
| Val | 150 | 15% |
| Test | 150 | 15% |

Split con `stratify=IsToxic` para mantener la proporción de clases en los tres conjuntos.

Genera: `data/processed/train.csv`, `val.csv`, `test.csv`

## 3. Modelo Baseline — TF-IDF + Logistic Regression

**Script**: `src/model/train.py`

### Pipeline sklearn

```python
Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1, 2))),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced')),
])
```

- `ngram_range=(1,2)`: captura unigramas y bigramas (ej: "not good" como feature)
- `class_weight='balanced'`: penaliza más los errores en la clase minoritaria
- Modelo guardado en `models/tfidf_lr.joblib`

### Resultados en validación

| Clase | Precision | Recall | F1 |
|---|---|---|---|
| No tóxico | 0.76 | 0.78 | 0.77 |
| Tóxico | 0.72 | 0.70 | 0.71 |
| **Accuracy** | | | **75%** |

### Resultados en test

| Clase | Precision | Recall | F1 |
|---|---|---|---|
| No tóxico | 0.71 | 0.74 | 0.73 |
| Tóxico | 0.74 | 0.70 | 0.72 |
| **Accuracy** | | | **72%** |

**Matriz de confusión (test)**:
```
Verdaderos negativos: 60  |  Falsos positivos: 21
Falsos negativos:     21  |  Verdaderos positivos: 48
```

## 4. Modelo DistilBERT — Fine-tuning

**Notebook de entrenamiento**: `notebooks/train_distilbert_colab.ipynb` (ejecutado en Google Colab con GPU T4)  
**Script de evaluación**: `src/model/evaluate_distilbert.py`

### Decisión

Se eligió DistilBERT sobre BERT completo por ser más ligero (66M parámetros vs 110M) manteniendo el 97% del rendimiento. El entrenamiento en local con CPU integrada de 128 MB no era viable (~40 min), por lo que se usó Google Colab.

### Configuración de entrenamiento

```python
TrainingArguments(
    num_train_epochs=3,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    metric_for_best_model='f1_toxic',   # métrica principal
    load_best_model_at_end=True,
)
```

- Modelo base: `distilbert-base-uncased`
- Tokenización: max_length=128, padding, truncation
- Mejor epoch: epoch 2 (F1 tóxico = 0.65 en val)

### Evolución del entrenamiento (val)

| Epoch | Loss train | Loss val | F1 tóxico |
|---|---|---|---|
| 1 | - | 0.619 | 0.600 |
| 2 | 0.646 | 0.577 | **0.650** ← mejor |
| 3 | 0.515 | 0.591 | 0.637 |

Epoch 3 sube ligeramente el loss de val → leve overfitting. Se guarda el modelo de epoch 2.

### Resultados en test

| Clase | Precision | Recall | F1 |
|---|---|---|---|
| No tóxico | 0.76 | 0.84 | 0.80 |
| Tóxico | 0.78 | 0.68 | 0.73 |
| **Accuracy** | | | **77%** |

**Matriz de confusión (test)**:
```
Verdaderos negativos: 68  |  Falsos positivos: 13
Falsos negativos:     22  |  Verdaderos positivos: 47
```

Modelo guardado en `models/distilbert_youtoxic/` (no incluido en git por tamaño, ver `models/README.md`).

## 5. Comparativa Final

| Métrica | TF-IDF + LR | DistilBERT |
|---|---|---|
| Accuracy | 72% | **77%** |
| F1 tóxico | 72% | **73%** |
| Precision tóxico | 74% | **78%** |
| Recall tóxico | **70%** | 68% |
| Falsos negativos | **21** | 22 |
| Falsos positivos | 21 | **13** |

### Conclusiones

- DistilBERT mejora en accuracy (+5%), precisión y falsos positivos
- El baseline tiene ligeramente mejor recall (detecta 1 tóxico más)
- Con solo 700 ejemplos de entrenamiento y 3 epochs, DistilBERT ya supera al baseline en la mayoría de métricas
- Para mejorar el recall de DistilBERT: más epochs, ajuste del umbral de decisión, o data augmentation

**Modelo seleccionado para producción**: DistilBERT (`models/distilbert_youtoxic/`)